# Data Fetch and Load

In [51]:
# import kagglehub
# kagglehub.login()

In [52]:
import shutil
import os
import pandas as pd
def fetch(): 

    if os.path.exists ('train.csv'):
        train_path = 'train.csv'
    if os.path.exists ('test.csv'):
        test_path = 'test.csv' 
    try: 
        return train_path, test_path
    except:
        data_path = kagglehub.competition_download('playground-series-s5e12')
        train_source = os.path.join (data_path, 'train.csv')
        test_source = os.path.join (data_path, 'test.csv')
        train_path = os.path.join (os.getcwd(), 'train.csv')
        test_path = os.path.join (os.getcwd(), 'test.csv')
        shutil.copy2(train_source, train_path)
        shutil.copy2(test_source, test_path)
        return train_path, test_path
    
train_path, test_path = fetch()
data = pd.read_csv(train_path); data.name = 'data'
submission = pd.read_csv(test_path); submission.name = 'submission'
all_df = pd.concat([data, submission], axis = 0)

# Data Overview

In [ ]:
def check_nan (df: pd.DataFrame):
    count = df.isna().sum()
    percent = (count / df.isna().count()) * 100
    result = pd.DataFrame({
        'Count': count, 
        'Percent': percent,
    })
    result = result[result['Percent'] > 0] 
    if len(result) == 0: 
        return (f'No Nan Values in {df.name}')

    return result

print (check_nan(data))
print (check_nan(submission))

No Nan Values in data
No Nan Values in submission


In [ ]:
print (data.name, data.shape)
print (submission.name, submission.shape)

data (700000, 26)
submission (300000, 25)


Analysis:
* 700.000 instances is big -> allow using Deep Learning / Complicated ML model
* 300.000 testcases on submission file (alot)

In [ ]:
all_df.transpose()

,0,1,2,3,4,5,6,7,8,9,...,299990,299991,299992,299993,299994,299995,299996,299997,299998,299999
id,0,1,2,3,4,5,6,7,8,9,...,999990,999991,999992,999993,999994,999995,999996,999997,999998,999999
age,31,50,32,54,54,42,41,51,34,44,...,48,51,41,51,65,59,50,63,48,47
alcohol_consumption_per_week,1,2,3,3,1,1,2,3,2,1,...,1,2,1,5,3,3,2,1,3,1
physical_activity_minutes_per_week,45,73,158,77,55,100,148,102,44,36,...,67,35,44,96,122,185,25,252,72,75
diet_score,7.7,5.7,8.5,4.6,5.7,4.4,3.4,4.0,2.7,5.8,...,3.9,7.9,5.9,6.3,4.2,6.3,5.8,5.2,4.9,5.1
sleep_hours_per_day,6.8,6.5,7.4,7.0,6.2,6.4,5.6,7.3,7.0,5.7,...,7.0,9.0,6.1,6.3,7.3,7.3,7.8,7.5,6.9,8.0
screen_time_hours_per_day,6.1,5.8,9.1,9.2,5.1,5.3,3.7,5.5,7.9,6.6,...,6.9,10.3,4.6,8.8,5.0,4.4,4.5,8.5,1.8,2.5
bmi,33.4,23.8,24.1,26.6,28.8,25.5,27.9,27.1,22.6,29.3,...,26.5,24.0,27.7,27.4,26.2,22.8,29.6,25.1,27.7,26.1
waist_to_hip_ratio,0.93,0.83,0.83,0.83,0.9,0.84,0.89,0.83,0.81,0.88,...,0.86,0.86,0.96,0.82,0.93,0.81,0.93,0.77,0.89,0.84
systolic_bp,112,120,95,121,108,111,130,125,120,110,...,101,117,103,121,105,108,112,129,121,132


In [56]:
all_df.describe()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.00000,700000.000000
mean,499999.500000,50.381533,2.077596,83.866288,5.958338,7.000878,6.012296,25.876851,0.858838,116.318170,75.427451,70.131929,186.965747,53.820317,103.058923,123.218839,0.150457,0.182716,0.03116,0.623296
std,288675.278932,11.741245,1.053658,55.006110,1.468700,0.905763,2.034110,2.870822,0.038144,11.083525,6.863409,6.984826,17.254180,8.306235,19.501570,26.080040,0.357519,0.386434,0.17375,0.484560
min,0.000000,19.000000,1.000000,1.000000,0.100000,3.100000,0.600000,15.100000,0.680000,91.000000,51.000000,42.000000,107.000000,21.000000,51.000000,31.000000,0.000000,0.000000,0.00000,0.000000
25%,249999.750000,42.000000,1.000000,49.000000,5.000000,6.400000,4.600000,23.900000,0.830000,108.000000,71.000000,65.000000,174.000000,48.000000,89.000000,106.000000,0.000000,0.000000,0.00000,0.000000
50%,499999.500000,50.000000,2.000000,72.000000,6.000000,7.000000,6.000000,25.900000,0.860000,116.000000,75.000000,70.000000,187.000000,54.000000,103.000000,123.000000,0.000000,0.000000,0.00000,1.000000
75%,749999.250000,59.000000,3.000000,100.000000,7.000000,7.600000,7.400000,27.800000,0.880000,124.000000,80.000000,75.000000,199.000000,59.000000,117.000000,139.000000,0.000000,0.000000,0.00000,1.000000
max,999999.000000,89.000000,9.000000,748.000000,9.900000,9.900000,16.500000,38.400000,1.050000,170.000000,104.000000,101.000000,289.000000,91.000000,226.000000,290.000000,1.000000,1.000000,1.00000,1.000000
